<a href="https://colab.research.google.com/github/JakeOh/202605_BD57/blob/main/lab_ml/ml21_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM(Large Language Model, 거대 언어 모델)


*   시퀀스-시퀀스 작업(Sequence-to-Sequence)
    *   시퀀스 데이터(sequential data)를 입력으로 받아서 시퀀스 데이터를 출력하는 작업.
    *   자연어 처리(NLP, Natural Language Processing) 분야에서 요약, 번역 등의 작업.
    *   두 개의 (순환) 신경망 연결한 인코더-디코더(encoder-decoder) 구조가 널리 사용됨.
*   어텐션 메커니즘(Attention mechanism)
    *   인코더-디코더 구조에서 사용된 순환 신경망(RNN)의 성능을 향상시키기 위해서 고안.
    *   기존에는 인코더의 마지막 타입 스텝에서 출력한 은닉 상태만을 사용해서 디코더가 새로운 텍스트를 생성.
    *   어텐션 메커니즘은 모든 타임 스텝에서 인코더가 출력한 은닉 상태를 디코더가 참조할 수 있도록 고안.
    *   디코더 새로운 토큰을 생성할 때 인코더가 처리한 토큰들 중에서 어떤 토큰에 더 주의(attention)를 기울 지를 결정. 입력 토큰들마다 디코더가 중요도를 다르게 부여.
*   트랜스포머 모델(Transformer model)
    *   어텐션 메커니즘을 기반으로 해서 인코더-디코더 구조에서 순환층을 제거.
    *   인코더에서 한 번에 하나의 토큰을 처리하지 않고 입력 텍스트 전체를 한 번에 처리.
    *   핵심 구성 요소
        *   멀티 헤드 어텐션(multi-head attention)
        *   층 정규화(layer normalization)
        *   잔차 연결(residual connection)
        *   피드 포워드 네트워크(feed-forward network)


![Evolutionary Tree of Modern LLMs](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcR78F8lk4wMrnX0X53gV5xwxvral2rsBP3prdjQlQfZ9Q&s=10)


*   오른쪽 회색 가지
    *   Transformer 모델을 사용하지 않은 알고리즘.
    *   워드 임베딩 벡터를 만드는 알고리즘.
*   Encoder 기반 모델
    *   텍스트 (긍정/부정) 분류
    *   개체명 인식 - 텍스트에서 사람 이름, 지역 이름, 회사 이름 등의 고유 명사를 식별.
    *   BERT, RoBERTa, ALBERT, ...
*   Encoder-Decoder 기반 모델
    *   문서 요약, 번역, 질문-답변
    *   T5, BART, ...
*   Decoder 기반 모델
    *   텍스트 생성 - 챗봇, 질문-답변, 요약, 번역, ...
    *   디코더
        *   이전까지 생성한 텍스트를 입력받아서 다음 토큰을 예측하는 방식.
        *   인코더로부터 입력이 없으면 디코더는 아무것도 생성할 수 없음.
        *   이전에 생성한 텍스트인 것처럼 어떤 텍스트를 입력해주면 인코더 도움 없이 다음 토큰을 예측할 수 있음.
        *   프롬프트(prompt): 이전에 생성된 텍스트인 것처럼 전달하는 초기 텍스트.
    *   GPT-4, GPT-5, LLaMA, Claude, ...
    *   현재 가장 활발히 연구되는 LLM 분야.


# BART 모델을 사용한 문서 요약

Hugging Face: 오픈 소스 (Transformer 기반) LLM 모델들 제공. Transformer 모델들을 사용할 수 있는 패키지를 제공.

In [1]:
import transformers

In [2]:
transformers.__version__

'5.16.1'

*   현재 코랩에 설치된 transformers 패키지의 버전은 5.16.1
*   transformers 5.0.x 버전부터 pipeline의 task(작업) 중에 "summarization" task가 목록에서 삭제됨.
*   BART 모델 소개를 위해서 transformers 버전을 4.x 버전으로 낮춤.
    *    transformers 4.x 버전 중에서 가장 마지막 안정적 버전은 4.57.6

In [4]:
!pip install transformers==4.57.6

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the

In [1]:
import transformers

In [2]:
transformers.__version__

'4.57.6'

In [3]:
# 훈련이 끝난 모델을 다운로드
summarizer = transformers.pipeline(task='summarization')

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


sshleifer/distilbart-cnn-12-6 훈련이 모두 끝난 LLM 모델을 다운로드.

In [4]:
# (요약하기 전) 원본 텍스트
sample_text = '''
Mary Alison Frantz (September 27, 1903 – February 1, 1995) was an American archaeological photographer and Byzantine scholar. She was the official photographer of the excavations of the Agora of Athens, and a prolific photographer of ancient Greek sculpture, including the Parthenon frieze and works from the Temple of Zeus at Olympia.

Frantz was born in Minnesota. Following her father's early death, she lived briefly in Scotland, where she first took an interest in photography. She studied classics at Smith College in Massachusetts, graduating in 1924. She first visited Greece in 1925 and held a fellowship at the American School of Classical Studies at Athens (ASCSA) in 1929–1930. She carried out her doctoral research under Charles Rufus Morey, receiving her PhD from Columbia University in 1937. Frantz began working at the ASCSA's Agora excavations in January 1934. From 1935, she took on an increasing share of the excavation's photography, and was made its official photographer in 1939. She also took the first photographs of the Linear B tablets from the Mycenaean site of Pylos, images used for the first transcription of the tablets and consequently for the decipherment of Linear B. As part of her work in the Agora excavations, she excavated and restored the Church of the Holy Apostles, the site's last surviving Byzantine structure.

During the Second World War, Frantz joined the Office of Strategic Services (OSS). She worked as an assistant to Carl Blegen, another archaeologist turned agent, and gathered intelligence on European exiles in the United States. She served on an Allied commission to observe the Greek elections of 1946, worked for the US Information Service, and was subsequently the cultural attaché of the US embassy in Athens. In this capacity, she established the Fulbright Program in Greece.

Frantz left the Agora excavations in 1964. Her later work largely consisted of collaborations with archaeologists such as Gisela Richter, Martin Robertson, and Bernard Ashmole. Her publications included some of the earliest archaeological research into Ottoman Greece, as well as photography of archaic kore sculptures, Byzantine architecture, and artifacts from the Aegean Bronze Age. Her work on late antiquity and later periods is considered pioneering, and to have contributed to raising the scholarly standing of post-classical archaeology in Greece. She was considered among the foremost photographers of ancient Greek antiquities, and her work has been cited as a major influence on the scholarship and popular reception of classical Greece.
'''

In [5]:
# Transformer LLM 모델의 출력값
result = summarizer(sample_text)

In [6]:
print(result)

[{'summary_text': ' Mary Alison Frantz was an American archaeologist and Byzantine scholar . She was the official photographer of the excavations of the Agora of Athens . She also took the first photographs of the Linear B tablets from the Mycenaean site of Pylos . Her work has been cited as a major influence on scholarship and popular reception of classical Greece .'}]


In [7]:
print(type(result))  #> list

<class 'list'>


In [9]:
len(result)  #> list의 원소는 1개

1

In [11]:
print(type(result[0]))  #> dict

<class 'dict'>


In [12]:
result[0].keys()

dict_keys(['summary_text'])

In [13]:
# dict.get(key) --> value 리턴
result[0].get('summary_text')

' Mary Alison Frantz was an American archaeologist and Byzantine scholar . She was the official photographer of the excavations of the Agora of Athens . She also took the first photographs of the Linear B tablets from the Mycenaean site of Pylos . Her work has been cited as a major influence on scholarship and popular reception of classical Greece .'

# KoBART 모델을 사용한 한글 문서 요약

In [14]:
kor_summarizer = transformers.pipeline(task='summarization',
                                       model='EbanLee/kobart-summary-v3')

config.json: 0.00B [00:00, ?B/s]

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/692 [00:00<?, ?B/s]

Device set to use cpu


In [15]:
kor_sample_text = '''
러시아의 우크라이나 침공(러시아어: Вторжение России на Украину, 우크라이나어: Російське вторгнення в Україну)은 2022년 2월 24일 러시아가 우크라이나의 영토를 침공한 후 현재까지 이어지는 러시아와 우크라이나 사이의 전쟁으로 이는 러시아-우크라이나 전쟁의 일부이다. 2022년 2월 24일, 러시아는 우크라이나를 침공했다. 이는 2014년에 시작된 두 국가 간 분쟁이 크게 확대되어 제2차 세계 대전 이후 유럽에서 가장 규모가 크고 치명적인 전쟁이 되었다.[13][14][15] 이 전투로 인해 수십만 명의 군인 사상자와 수만 명의 우크라이나 민간인 사상자가 발생했다. 2025년 기준, 러시아군은 우크라이나 영토의 약 20%를 점령하고 있다. 4,100만 명의 인구 중 약 800만 명의 우크라이나인이 국내에서 피난했으며, 2023년 4월까지 820만 명 이상이 국외로 피난하여 제2차 세계 대전 이후 유럽의 가장 큰 난민 위기를 초래했다.

2021년 후반, 러시아는 우크라이나 국경 근처에 병력을 집결시키고 서방 세계에 우크라이나가 북대서양 조약 기구(NATO) 군사 동맹에 가입하는 것을 금지하는 것을 포함한 요구사항을 발표했다.[16] 우크라이나 공격 계획을 반복적으로 부인한 후, 2022년 2월 24일, 러시아 대통령 블라디미르 푸틴은 '특별 군사 작전'을 발표했다. 그는 이 작전이 러시아가 지원하는 분리주의 도네츠크 및 루한스크 공화국을 지원하기 위한 것이라고 말했다. 이들의 준군사 조직은 2014년부터 돈바스 전쟁에서 우크라이나와 싸워왔다. 푸틴은 우크라이나의 국가 정당성에 도전하는 민족통일주의 및 제국주의적 견해를 표명하며, 우크라이나 정부가 네오나치이며 돈바스에서 러시아 소수민족에 대한 집단학살을 자행하고 있다고 근거 없이 주장했다. 그리고 러시아의 목표는 우크라이나를 "비무장화 및 탈나치화"하는 것이라고 말했다.[17][18][19][20] 러시아의 공습과 지상 침공은 수도 키이우를 향한 북부 전선에서 벨라루스로부터, 크림반도로부터 남부 전선에서, 그리고 돈바스 및 하르키우를 향한 동부 전선에서 개시되었다. 우크라이나는 계엄령을 발동하고, 총동원령을 내렸으며, 러시아와의 외교 관계를 단절했다.

러시아군은 격렬한 저항과 병참 문제에 직면한 후 2022년 4월까지 북부와 키이우 외곽에서 퇴각했다. 그들의 철수 후 부차 학살이 드러났다. 남동부에서 러시아는 돈바스 공세를 개시하고 파괴적인 포위 끝에 마리우폴을 점령했다. 러시아는 전선에서 멀리 떨어진 군사 및 민간 목표물에 대한 폭격을 계속했으며, 겨울철에는 에너지 그리드를 공격했다. 2022년 후반, 우크라이나는 남부와 동부에서 성공적인 반격 작전을 개시하여 하르키우주 대부분을 해방했다. 직후 러시아는 부분적으로 점령한 4개 주를 불법적으로 합병했다. 11월에는 우크라이나가 헤르손을 해방했다. 2023년 6월, 우크라이나는 남동부에서 또 다른 반격을 개시했지만 거의 진전을 이루지 못했다. 2024년 상반기 동안 동부에서 러시아가 작지만 꾸준히 진격한 후, 우크라이나는 8월에 러시아 쿠르스크주로 국경을 넘는 공세를 시작했고, 그곳에는 북한 군인들이 러시아를 돕기 위해 파견되었다. 유엔 인권최고대표사무소는 러시아가 점령된 우크라이나에서 심각한 인권 침해를 자행하고 있다고 보고한다. 전쟁으로 인한 러시아의 직접 비용은 4,500억 달러를 초과했다.[21][22][23]

이번 침공은 광범위한 국제적 비난을 받았다. 유엔 총회는 침공을 비난하고 러시아의 완전한 철수를 요구하는 결의안을 통과시켰다. 국제사법재판소는 러시아에 군사 작전을 중단하라고 명령했으며, 유럽 평의회는 러시아를 추방했다. 많은 국가들이 러시아와 그 동맹국인 벨라루스에 제재를 부과하고 우크라이나에 대규모 인도주의적 및 군사적 지원을 제공했다. 발트해 국가들과 폴란드는 러시아를 테러 국가로 선언했다. 전 세계적으로 시위가 발생했으며, 러시아의 반전 시위자들은 대규모 체포와 심한 언론 검열에 직면했다. 러시아의 민간인 공격은 집단학살 혐의로 이어졌다.[24][25][26][27] 전쟁 관련 우크라이나 농업 및 해운의 혼란은 세계 식량 위기를 초래했으며, 전쟁 관련 지역 환경 피해는 생태학살로 묘사되었고, 전쟁은 전 세계 기후 정책을 심각하게 방해했다. 국제형사재판소(ICC)는 인도에 반한 죄 및 전쟁 범죄, 우크라이나 어린이 납치, 우크라인에 대한 집단학살에 대한 조사를 개시, 푸틴과 다른 5명의 러시아 관리들에 대한 체포 영장을 발부했다.
'''

In [16]:
result = kor_summarizer(kor_sample_text)

In [18]:
print(type(result))

<class 'list'>


In [19]:
print(type(result[0]))

<class 'dict'>


In [20]:
result[0].keys()

dict_keys(['summary_text'])

In [21]:
result[0].get('summary_text')

'러시아가 우크라이나 영토를 침공한 후 현재까지 이어지는 러시아와 우크라이나 사이의 전쟁으로 이는 러시아-우크라이나 전쟁의 일부이다. 2022년 2월 24일, 러시아는 우크라이나를 침공했고 이로 인해 수십만 명의 군인 사상자와 수만 명의 우크라이나 민간인 사상자가 발생했다. 2021년 후반, 러시아는 우크라이나가 북대서양 조약 기구(NATO) 군사 동맹에 가입하는 것을 금지하는 것을 포함한 요구사항을 발표했다. 러시아군은 격렬한 저항과 병참 문제에 직면한 후 북부와 키이우 외곽에서 퇴각했다. 남동부에서 러시아는 돈바스 공세를 개시하고 파괴적인 포위 끝에 마리우폴을 점령했다. 러시아는 전선에서 멀리 떨어진 군사 및 민간 목표물에 대한 폭격을 계속했으며, 겨울철에는 에너지 그리드를 공격했다.'